# Séance 10 · Exercices — Parler à un LLM par le code · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab**, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
Tous les exercices marchent en **mode démo** (`USE_MODEL = False`) : ce qu'on vérifie surtout, c'est la forme des messages, le prompt système, l'historique et le parsing du JSON, pas le talent du modèle.


## Préparation

La même cellule qu'à la leçon (elle prépare `llm(messages)`), plus `tiktoken` pour compter les tokens de l'historique et le helper `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots du prompt (mode démo)."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    tout = " ".join(m["content"] for m in messages).lower()
    if "json" in ql or "json" in systeme:                 # section 5 : sortie structurée
        if "quiz" in tout or "question" in tout:
            return ('Voici le quiz : {"question": "Quel est le type de Pikachu ?", '
                    '"choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"}')
        if "pok" in tout:
            return '{"nom": "Pikachu", "type": "Électrique", "pv": 35, "attaque_preferee": "Éclair"}'
        return '{"reponse": "Paris", "confiance": 0.9}'
    prenom = re.search(r"je m'appelle (\w+)", tout)
    if "prénom" in ql or "souviens" in ql:                # section 4 : l'historique
        return f"Bien sûr, tu t'appelles {prenom.group(1).capitalize()} !" if prenom else "Tu ne me l'as pas encore dit !"
    if "pirate" in systeme:
        return "Arrr ! Moussaillon, hisse tes cahiers et cap sur les fractions, le trésor est au bout !"
    if "maître du jeu" in systeme or "mj" in systeme:
        return "Tu entres dans la taverne. Un nain te fixe et pose une carte sur la table. Que fais-tu ? (1) lui parler (2) prendre la carte"
    if "quiz" in systeme:
        return "Question 1 : quel est le type de Salamèche ? A) Eau B) Feu C) Plante"
    if "coach" in systeme:
        return "Super, on y va ! Commence par 25 minutes de révision, puis 5 minutes de pause. Sur quelle matière on attaque ?"
    if "capitale" in ql:
        return "La capitale de la France est Paris."
    if "bonjour" in ql or "salut" in ql:
        return "Salut ! Je suis là pour t'aider. Qu'est-ce qu'on fait aujourd'hui ?"
    return "Bonne question ! En résumé : c'est un sujet intéressant, et je peux t'en dire plus si tu veux."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

# ---------- Outils des exercices ----------
try:
    import tiktoken
except ImportError:
    %pip install -q tiktoken
    import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

ROLES_VALIDES = {"system", "user", "assistant"}
SYSTEME_JSON = "Tu réponds UNIQUEMENT avec un objet JSON valide, sans texte autour, sans explication."

# ---------- Vérification automatique des exercices ----------
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Prêt.")

## Exercice 1 ⭐ · Ton premier message

Construis la liste `messages` avec **un seul** message de rôle `user` qui demande la capitale de la France, puis envoie-la au modèle avec `llm(messages)` et range la réponse dans `reponse`.

Résultat attendu : une liste d'un dictionnaire `{"role": ..., "content": ...}`, et une réponse (texte) qui contient « Paris ».

<details><summary>Indice</summary>

Un message = un dictionnaire avec exactement deux clés : `"role"` et `"content"`. La liste : `[{"role": "user", "content": "..."}]`.

</details>

In [ ]:
# À toi
messages = None
reponse = None

print(reponse)

In [ ]:
verifier("Exercice 1 · un message user", lambda: isinstance(messages, list) and len(messages) == 1
         and messages[0]["role"] == "user" and "capitale" in messages[0]["content"].lower())
verifier("Exercice 1 · réponse du modèle", lambda: isinstance(reponse, str) and "paris" in reponse.lower())

<details><summary>Solution</summary>

```python
messages = [{"role": "user", "content": "Quelle est la capitale de la France ? Réponds en une phrase."}]
reponse = llm(messages)
print(reponse)
```

</details>

## Exercice 2 ⭐ · Chasse aux erreurs

Écris `est_valide(messages)` qui renvoie `True` si une liste de messages est correcte : c'est une **liste**, chaque élément est un **dictionnaire** avec exactement les clés `role` et `content`, le rôle est `system`, `user` ou `assistant`, et le contenu est une chaîne non vide. Teste-la sur les 6 candidats.

Résultat attendu : `[True, False, False, False, True, False]`.

<details><summary>Indice</summary>

Commence par `if not isinstance(messages, list): return False`, puis une boucle `for m in messages` qui renvoie `False` dès qu'une règle est cassée. `set(m) == {"role", "content"}` vérifie les clés.

</details>

In [ ]:
# À toi
def est_valide(messages):
    return None

candidats = [
    [{"role": "user", "content": "Salut"}],
    {"role": "user", "content": "Salut"},                                          # pas une liste
    [{"role": "utilisateur", "content": "Salut"}],                                 # rôle inconnu
    [{"role": "user", "text": "Salut"}],                                           # mauvaise clé
    [{"role": "system", "content": "Tu es sympa."}, {"role": "user", "content": "Salut"}],
    [{"role": "user", "content": 42}],                                             # contenu pas texte
]
resultats = [est_valide(c) for c in candidats]
print(resultats)

In [ ]:
verifier("Exercice 2 · est_valide", resultats == [True, False, False, False, True, False])
verifier("Exercice 2 · contenu vide refusé", lambda: est_valide([{"role": "user", "content": ""}]) is False)

<details><summary>Solution</summary>

```python
def est_valide(messages):
    if not isinstance(messages, list):
        return False
    for m in messages:
        if not isinstance(m, dict) or set(m) != {"role", "content"}:
            return False
        if m["role"] not in ROLES_VALIDES:
            return False
        if not isinstance(m["content"], str) or m["content"] == "":
            return False
    return True

candidats = [
    [{"role": "user", "content": "Salut"}],
    {"role": "user", "content": "Salut"},
    [{"role": "utilisateur", "content": "Salut"}],
    [{"role": "user", "text": "Salut"}],
    [{"role": "system", "content": "Tu es sympa."}, {"role": "user", "content": "Salut"}],
    [{"role": "user", "content": 42}],
]
resultats = [est_valide(c) for c in candidats]
print(resultats)   # [True, False, False, False, True, False]
```

</details>

## Exercice 3 ⭐ · System + user

Écris `construire(question, systeme)` qui renvoie la liste de deux messages (d'abord le `system`, puis le `user`), et `demander(question, systeme=DEFAUT)` qui envoie cette liste au modèle et renvoie sa réponse.

Résultat attendu : `construire("Q ?", "S")` → `[{"role": "system", "content": "S"}, {"role": "user", "content": "Q ?"}]`.

<details><summary>Indice</summary>

`demander` tient en une ligne : `return llm(construire(question, systeme))`.

</details>

In [ ]:
# À toi
DEFAUT = "Tu es un assistant sympa qui répond en français, en 3 phrases maximum."

def construire(question, systeme):
    return None

def demander(question, systeme=DEFAUT):
    return None

print(construire("Quelle est la capitale de la France ?", DEFAUT))
print(demander("Quelle est la capitale de la France ?"))

In [ ]:
verifier("Exercice 3 · construire", lambda: construire("Q ?", "S") == [{"role": "system", "content": "S"}, {"role": "user", "content": "Q ?"}])
verifier("Exercice 3 · demander", lambda: "paris" in demander("Quelle est la capitale de la France ?").lower())

<details><summary>Solution</summary>

```python
DEFAUT = "Tu es un assistant sympa qui répond en français, en 3 phrases maximum."

def construire(question, systeme):
    return [{"role": "system", "content": systeme}, {"role": "user", "content": question}]

def demander(question, systeme=DEFAUT):
    return llm(construire(question, systeme))

print(construire("Quelle est la capitale de la France ?", DEFAUT))
print(demander("Quelle est la capitale de la France ?"))
```

</details>

## Exercice 4 ⭐ · La fiche de poste (prompt système)

Écris un prompt système complet pour un assistant de ton choix, avec les 4 éléments de la leçon : **qui** (« Tu es ... »), **comment** (ton, langue, longueur), **quoi faire**, **quoi ne pas faire** (« Tu ne ... jamais »). Au moins 25 mots. Teste-le ensuite avec `demander`.

Résultat attendu : la vérification cherche « Tu es », « français » et une interdiction (« jamais », « ne ... pas », « interdit »).

<details><summary>Indice</summary>

Inspire-toi de l'exemple de la leçon : « Tu es un coach de révisions. Tu tutoies, tu réponds en français en 3 phrases maximum. Tu poses toujours une question à la fin. Tu ne donnes jamais la réponse directement : tu donnes un indice. »

</details>

In [ ]:
# À toi
mon_systeme = """À compléter"""

reponse = demander("J'ai un contrôle de maths demain et je stresse.", systeme=mon_systeme)
print(reponse)

In [ ]:
s = mon_systeme.lower()
verifier("Exercice 4 · qui (« Tu es »)", "tu es" in s)
verifier("Exercice 4 · comment (langue)", "français" in s)
verifier("Exercice 4 · quoi ne pas faire", lambda: re.search(r"jamais|ne \w+ pas|interdit|pas le droit", s) is not None)
verifier("Exercice 4 · au moins 25 mots", len(mon_systeme.split()) >= 25)

Check-list (à l'œil) : la réponse du modèle respecte-t-elle le ton demandé ? La langue ? La longueur ? Une règle qu'il a oubliée ?

<details><summary>Solution</summary>

```python
mon_systeme = """Tu es un coach de révisions bienveillant. Tu tutoies, tu réponds en français, en 3 phrases maximum,
avec un ton encourageant. Tu aides à organiser les révisions et tu poses toujours une question à la fin.
Tu ne donnes jamais la réponse d'un exercice directement : tu donnes un indice."""

reponse = demander("J'ai un contrôle de maths demain et je stresse.", systeme=mon_systeme)
print(reponse)
```

</details>

## Exercice 5 ⭐⭐ · Traduire une conversation en messages

On a noté une conversation sous forme de couples `(qui, texte)` avec `"système"`, `"moi"` et `"bot"`. Écris `vers_messages(transcription)` qui la transforme en liste de messages avec les bons rôles (`system`, `user`, `assistant`), puis envoie-la au modèle : il doit se souvenir du prénom.

Résultat attendu : les rôles `["system", "user", "assistant", "user"]`, et une réponse qui contient « Nova ».

<details><summary>Indice</summary>

Un dictionnaire de correspondance `ROLES = {"système": "system", "moi": "user", "bot": "assistant"}` et une compréhension de liste.

</details>

In [ ]:
# À toi
transcription = [
    ("système", "Tu es un coach sportif très motivé. Tu réponds en français, en 2 phrases maximum."),
    ("moi", "Salut, je m'appelle Nova et je veux progresser en course à pied."),
    ("bot", "Salut Nova ! On commence par 20 minutes de footing trois fois par semaine."),
    ("moi", "Tu te souviens de mon prénom ?"),
]

def vers_messages(transcription):
    return None

messages = vers_messages(transcription)
reponse = None
print(reponse)

In [ ]:
verifier("Exercice 5 · rôles", lambda: [m["role"] for m in messages] == ["system", "user", "assistant", "user"])
verifier("Exercice 5 · contenus conservés", lambda: [m["content"] for m in messages] == [t for _, t in transcription] and est_valide(messages))
verifier("Exercice 5 · le modèle se souvient", lambda: "nova" in reponse.lower())

<details><summary>Solution</summary>

```python
transcription = [
    ("système", "Tu es un coach sportif très motivé. Tu réponds en français, en 2 phrases maximum."),
    ("moi", "Salut, je m'appelle Nova et je veux progresser en course à pied."),
    ("bot", "Salut Nova ! On commence par 20 minutes de footing trois fois par semaine."),
    ("moi", "Tu te souviens de mon prénom ?"),
]
ROLES = {"système": "system", "moi": "user", "bot": "assistant"}

def vers_messages(transcription):
    return [{"role": ROLES[qui], "content": texte} for qui, texte in transcription]

messages = vers_messages(transcription)
reponse = llm(messages)
print(reponse)   # il « se souvient » parce qu'on lui remontre toute la conversation
```

</details>

## Exercice 6 ⭐⭐ · Un chatbot qui se souvient

Complète la méthode `parler` de la classe `Chatbot` : elle ajoute le message de l'utilisateur à l'historique, appelle `llm(self.historique)`, ajoute la réponse avec le rôle `assistant`, et renvoie la réponse. Puis tiens une conversation de 2 tours avec un guide touristique.

Résultat attendu : après 2 tours, l'historique contient 5 messages (system, user, assistant, user, assistant) et le bot se souvient du prénom « Lina ».

<details><summary>Indice</summary>

Trois `append` et un `return` : `self.historique.append({"role": "user", "content": message})`, puis `reponse = llm(self.historique)`, puis l'append de l'assistant.

</details>

In [ ]:
# À toi
class Chatbot:
    """Un chatbot = un prompt système + l'historique des messages."""

    def __init__(self, systeme):
        self.historique = [{"role": "system", "content": systeme}]

    def parler(self, message):
        return None

bot = Chatbot("Tu es un guide touristique de Paris, enthousiaste. Tu réponds en français, en 2 phrases.")
r1 = bot.parler("Bonjour ! Je m'appelle Lina.")
r2 = bot.parler("Tu te souviens de mon prénom ?")
print(r1)
print(r2)
print("Messages dans l'historique :", len(bot.historique))

In [ ]:
verifier("Exercice 6 · 5 messages", lambda: [m["role"] for m in bot.historique] == ["system", "user", "assistant", "user", "assistant"])
verifier("Exercice 6 · la réponse est rangée", lambda: bot.historique[2]["content"] == r1 and bot.historique[4]["content"] == r2)
verifier("Exercice 6 · il se souvient de Lina", lambda: "lina" in r2.lower())

<details><summary>Solution</summary>

```python
class Chatbot:
    """Un chatbot = un prompt système + l'historique des messages."""

    def __init__(self, systeme):
        self.historique = [{"role": "system", "content": systeme}]

    def parler(self, message):
        self.historique.append({"role": "user", "content": message})
        reponse = llm(self.historique)
        self.historique.append({"role": "assistant", "content": reponse})
        return reponse

bot = Chatbot("Tu es un guide touristique de Paris, enthousiaste. Tu réponds en français, en 2 phrases.")
r1 = bot.parler("Bonjour ! Je m'appelle Lina.")
r2 = bot.parler("Tu te souviens de mon prénom ?")
print(r1)
print(r2)
print("Messages dans l'historique :", len(bot.historique))   # 5
```

</details>

## Exercice 7 ⭐⭐ · Une mémoire qui déborde

Tout l'historique repart au modèle à chaque tour : plus il est long, plus ça coûte. Écris :
1. `compter_tokens(historique)` : la somme des tokens (`tiktoken`) du `content` de chaque message ;
2. `tronquer(historique, n_tours)` : garde le message `system` et seulement les **`n_tours` derniers échanges** (un échange = un message `user` + un `assistant`, donc `2 * n_tours` messages), sans modifier la liste d'origine.

Résultat attendu : sur `long_historique` (1 system + 6 échanges), `tronquer(long_historique, 2)` a 5 messages, commence par le system, puis un `user`, et finit par le même message que l'original.

<details><summary>Indice</summary>

`historique[:1] + historique[-2 * n_tours:]`. Pour les tokens : `sum(len(enc.encode(m["content"])) for m in historique)`.

</details>

In [ ]:
# À toi
long_historique = [{"role": "system", "content": "Tu es un guide touristique de Paris. Tu réponds en 2 phrases."}]
for i in range(1, 7):
    long_historique.append({"role": "user", "content": f"Question numéro {i} : que visiter aujourd'hui ?"})
    long_historique.append({"role": "assistant", "content": f"Réponse numéro {i} : la tour Eiffel, puis le Louvre et une glace sur les quais."})

def compter_tokens(historique):
    return None

def tronquer(historique, n_tours):
    return None

print("Tokens avant :", compter_tokens(long_historique), "| après tronquage à 2 tours :", compter_tokens(tronquer(long_historique, 2)))

In [ ]:
verifier("Exercice 7 · compter_tokens", lambda: compter_tokens(long_historique) == sum(len(enc.encode(m["content"])) for m in long_historique))
court = tronquer(long_historique, 2)
verifier("Exercice 7 · tronquer garde system + 2 tours", lambda: len(court) == 5 and court[0]["role"] == "system" and court[1]["role"] == "user" and court[-1] == long_historique[-1])
verifier("Exercice 7 · l'original est intact", lambda: len(long_historique) == 13 and compter_tokens(court) < compter_tokens(long_historique))

<details><summary>Solution</summary>

```python
long_historique = [{"role": "system", "content": "Tu es un guide touristique de Paris. Tu réponds en 2 phrases."}]
for i in range(1, 7):
    long_historique.append({"role": "user", "content": f"Question numéro {i} : que visiter aujourd'hui ?"})
    long_historique.append({"role": "assistant", "content": f"Réponse numéro {i} : la tour Eiffel, puis le Louvre et une glace sur les quais."})

def compter_tokens(historique):
    return sum(len(enc.encode(m["content"])) for m in historique)

def tronquer(historique, n_tours):
    return historique[:1] + historique[-2 * n_tours:]     # le system + les 2·n derniers messages (nouvelle liste)

print("Tokens avant :", compter_tokens(long_historique), "| après tronquage à 2 tours :", compter_tokens(tronquer(long_historique, 2)))
# Les vrais chatbots font pareil (ou résument les vieux messages) pour tenir dans la fenêtre du modèle.
```

</details>

## Exercice 8 ⭐⭐ · Extraire le JSON d'une réponse

Le modèle ajoute souvent du texte autour du JSON, et parfois le JSON est cassé. Écris `extraire_json(texte)` qui prend la partie entre la **première `{`** et la **dernière `}`**, la lit avec `json.loads`, et renvoie le dictionnaire, ou `None` si c'est impossible (pas d'accolades, ou JSON illisible).

Résultat attendu : un dictionnaire pour les deux premiers essais, `None` pour les deux derniers.

<details><summary>Indice</summary>

`debut, fin = texte.find("{"), texte.rfind("}")`. Si l'un des deux vaut `-1` → `None`. Puis `try: return json.loads(texte[debut:fin + 1]) except json.JSONDecodeError: return None`.

</details>

In [ ]:
# À toi
def extraire_json(texte):
    return None

essais = [
    'Voici le quiz : {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"} Bonne chance !',
    '{"nom": "Pikachu", "pv": 35}',
    "Désolé, je ne peux pas répondre en JSON.",
    '{"nom": "Pikachu", "pv": }',
]
for e in essais:
    print(extraire_json(e))

In [ ]:
verifier("Exercice 8 · JSON avec du texte autour", lambda: extraire_json(essais[0])["bonne_reponse"] == "Électrique" and len(extraire_json(essais[0])["choix"]) == 3)
verifier("Exercice 8 · JSON propre", lambda: extraire_json(essais[1]) == {"nom": "Pikachu", "pv": 35})
verifier("Exercice 8 · pas de JSON → None", lambda: extraire_json(essais[2]) is None and extraire_json(essais[3]) is None)

<details><summary>Solution</summary>

```python
def extraire_json(texte):
    debut, fin = texte.find("{"), texte.rfind("}")
    if debut == -1 or fin == -1:
        return None
    try:
        return json.loads(texte[debut:fin + 1])
    except json.JSONDecodeError:
        return None

essais = [
    'Voici le quiz : {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"} Bonne chance !',
    '{"nom": "Pikachu", "pv": 35}',
    "Désolé, je ne peux pas répondre en JSON.",
    '{"nom": "Pikachu", "pv": }',
]
for e in essais:
    print(extraire_json(e))
```

</details>

## Exercice 9 ⭐⭐ · Redemander quand ça rate

Écris `demander_json(question, systeme, essais=3, modele=llm)` : elle appelle `modele([...system..., ...user...])`, essaie `extraire_json` sur la réponse, et **redemande** (jusqu'à `essais` fois) tant que le résultat est `None`. Après le dernier échec, elle renvoie `None`. Le paramètre `modele` permet de tester avec un faux modèle qui ne répond jamais en JSON.

Résultat attendu : avec `llm_casse`, le résultat est `None` et le modèle a été appelé exactement 3 fois ; avec `llm`, on obtient un dictionnaire avec la clé `nom`.

<details><summary>Indice</summary>

`for essai in range(1, essais + 1): texte = modele(messages); resultat = extraire_json(texte); if resultat is not None: return resultat; print(f"essai {essai} : JSON illisible, on redemande")`. Après la boucle : `return None`.

</details>

In [ ]:
# À toi
def demander_json(question, systeme, essais=3, modele=llm):
    return None

appels = []                                    # pour compter les appels au faux modèle
def llm_casse(messages, **options):
    appels.append(1)
    return "Oups, pas de JSON ici."

resultat_casse = demander_json("Donne une fiche de Pikachu.", SYSTEME_JSON, modele=llm_casse)
fiche = demander_json("Donne une fiche du Pokémon Pikachu en JSON avec les clés : nom, type, pv.", SYSTEME_JSON)
print("faux modèle :", resultat_casse, "| appels :", len(appels))
print("vrai modèle :", fiche)

In [ ]:
verifier("Exercice 9 · 3 essais puis None", lambda: resultat_casse is None and len(appels) == 3)
verifier("Exercice 9 · fiche JSON lue", lambda: isinstance(fiche, dict) and "nom" in fiche)

<details><summary>Solution</summary>

```python
def demander_json(question, systeme, essais=3, modele=llm):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    for essai in range(1, essais + 1):
        texte = modele(messages)
        resultat = extraire_json(texte)
        if resultat is not None:
            return resultat
        print(f"essai {essai} : JSON illisible → on redemande")
    return None

appels = []
def llm_casse(messages, **options):
    appels.append(1)
    return "Oups, pas de JSON ici."

resultat_casse = demander_json("Donne une fiche de Pikachu.", SYSTEME_JSON, modele=llm_casse)
fiche = demander_json("Donne une fiche du Pokémon Pikachu en JSON avec les clés : nom, type, pv.", SYSTEME_JSON)
print("faux modèle :", resultat_casse, "| appels :", len(appels))   # None | 3
print("vrai modèle :", fiche)
```

</details>

## Exercice 10 ⭐⭐⭐ · Valider la structure d'un quiz

Un JSON lisible n'est pas forcément **le bon** JSON. Écris `valider_quiz(q)` qui renvoie `True` seulement si `q` est un dictionnaire avec : `question` (texte non vide), `choix` (une liste de **3** textes) et `bonne_reponse` (un texte qui est **dans** `choix`).

Résultat attendu : `[True, False, False, False, False, False]`.

<details><summary>Indice</summary>

Vérifie dans l'ordre : `isinstance(q, dict)`, les 3 clés présentes, les types, `len(q["choix"]) == 3`, `q["bonne_reponse"] in q["choix"]`. Renvoie `False` dès qu'une règle casse.

</details>

In [ ]:
# À toi
def valider_quiz(q):
    return None

quiz_candidats = [
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"]},                                  # clé manquante
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique"], "bonne_reponse": "Électrique"},           # 2 choix
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Plante", "Eau"], "bonne_reponse": "Électrique"},        # réponse absente des choix
    {"question": "Quel est le type de Pikachu ?", "choix": "Feu, Électrique, Eau", "bonne_reponse": "Électrique"},          # choix pas une liste
    None,                                                                                                                # pas de JSON du tout
]
print([valider_quiz(q) for q in quiz_candidats])

In [ ]:
verifier("Exercice 10 · valider_quiz", lambda: [valider_quiz(q) for q in quiz_candidats] == [True, False, False, False, False, False])
verifier("Exercice 10 · question vide refusée", lambda: valider_quiz({"question": "", "choix": ["a", "b", "c"], "bonne_reponse": "a"}) is False)

<details><summary>Solution</summary>

```python
def valider_quiz(q):
    if not isinstance(q, dict):
        return False
    if not all(cle in q for cle in ["question", "choix", "bonne_reponse"]):
        return False
    if not isinstance(q["question"], str) or not q["question"]:
        return False
    if not isinstance(q["choix"], list) or len(q["choix"]) != 3 or not all(isinstance(c, str) for c in q["choix"]):
        return False
    return isinstance(q["bonne_reponse"], str) and q["bonne_reponse"] in q["choix"]

quiz_candidats = [
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"]},
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Plante", "Eau"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Pikachu ?", "choix": "Feu, Électrique, Eau", "bonne_reponse": "Électrique"},
    None,
]
print([valider_quiz(q) for q in quiz_candidats])   # [True, False, False, False, False, False]
```

</details>

## Exercice 11 ⭐⭐⭐ · Générer un quiz structuré

Demande au modèle une question de quiz sur les Pokémon en JSON (clés `question`, `choix` liste de 3, `bonne_reponse`) avec `demander_json`, vérifie-la avec `valider_quiz`, puis écris `afficher_quiz(q)` qui **renvoie** un texte : la question sur la première ligne, les choix numérotés `1.`, `2.`, `3.` sur les lignes suivantes, et `Bonne réponse : ...` sur la dernière ligne.

Résultat attendu (mode démo) : un quiz valide sur le type de Pikachu. Avec le vrai petit modèle, il arrive que le JSON ne soit pas valide : c'est justement pour ça qu'on valide.

<details><summary>Indice</summary>

`lignes = [q["question"]] + [f"{i}. {c}" for i, c in enumerate(q["choix"], 1)] + [f"Bonne réponse : {q['bonne_reponse']}"]`, puis `"\n".join(lignes)`.

</details>

In [ ]:
# À toi
quiz = demander_json("Écris une question de quiz sur les Pokémon, en JSON avec les clés : question, choix (liste de 3), bonne_reponse.", SYSTEME_JSON)

def afficher_quiz(q):
    return None

quiz_exemple = {"question": "Quel est le type de Salamèche ?", "choix": ["Eau", "Feu", "Plante"], "bonne_reponse": "Feu"}
print(afficher_quiz(quiz_exemple))
print()
print("Quiz du modèle valide :", valider_quiz(quiz))
print(afficher_quiz(quiz) if valider_quiz(quiz) else quiz)

In [ ]:
lignes = (afficher_quiz(quiz_exemple) or "").split("\n")
verifier("Exercice 11 · afficher_quiz", lambda: lignes[0] == quiz_exemple["question"] and lignes[1:4] == ["1. Eau", "2. Feu", "3. Plante"] and lignes[-1] == "Bonne réponse : Feu")
verifier("Exercice 11 · quiz du modèle valide (mode démo)", lambda: valider_quiz(quiz))

<details><summary>Solution</summary>

```python
quiz = demander_json("Écris une question de quiz sur les Pokémon, en JSON avec les clés : question, choix (liste de 3), bonne_reponse.", SYSTEME_JSON)

def afficher_quiz(q):
    lignes = [q["question"]]
    lignes += [f"{i}. {c}" for i, c in enumerate(q["choix"], 1)]
    lignes.append(f"Bonne réponse : {q['bonne_reponse']}")
    return "\n".join(lignes)

quiz_exemple = {"question": "Quel est le type de Salamèche ?", "choix": ["Eau", "Feu", "Plante"], "bonne_reponse": "Feu"}
print(afficher_quiz(quiz_exemple))
print()
print("Quiz du modèle valide :", valider_quiz(quiz))
print(afficher_quiz(quiz) if valider_quiz(quiz) else quiz)
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : une partie de quiz avec score

Combine tout : une banque de 3 quiz (déjà validés), un joueur qui répond soit par le **texte** du choix (majuscules ignorées) soit par son **numéro** (`"2"`), et un bilan en JSON.
1. `corriger(q, reponse_joueur)` renvoie `True` si la réponse est bonne ;
2. `jouer(banque, reponses)` renvoie `{"score": ..., "total": ..., "details": [True/False, ...]}` ;
3. `bilan_json` = ce dictionnaire transformé en texte JSON avec `json.dumps` (puis relisible avec `json.loads`).

Résultat attendu : le joueur ci-dessous marque **2 / 3**.

<details><summary>Indice</summary>

Pour le numéro : `if reponse.strip().isdigit(): reponse = q["choix"][int(reponse) - 1]`. Puis `reponse.strip().lower() == q["bonne_reponse"].lower()`.

</details>

In [ ]:
# À toi
banque = [
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Carapuce ?", "choix": ["Feu", "Eau", "Plante"], "bonne_reponse": "Eau"},
    {"question": "Quel Pokémon est de type Feu ?", "choix": ["Bulbizarre", "Salamèche", "Carapuce"], "bonne_reponse": "Salamèche"},
]
reponses_joueur = ["électrique", "2", "Eau"]

def corriger(q, reponse_joueur):
    return None

def jouer(banque, reponses):
    return None

bilan = jouer(banque, reponses_joueur)
bilan_json = None
print(bilan_json)

In [ ]:
verifier("Exercice 12 · corriger (texte, numéro, faux)", lambda: corriger(banque[0], "électrique") is True and corriger(banque[1], "2") is True and corriger(banque[2], "Eau") is False)
verifier("Exercice 12 · jouer", lambda: bilan["score"] == 2 and bilan["total"] == 3 and bilan["details"] == [True, True, False])
verifier("Exercice 12 · bilan_json relisible", lambda: json.loads(bilan_json)["score"] == 2)

<details><summary>Solution</summary>

```python
banque = [
    {"question": "Quel est le type de Pikachu ?", "choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"},
    {"question": "Quel est le type de Carapuce ?", "choix": ["Feu", "Eau", "Plante"], "bonne_reponse": "Eau"},
    {"question": "Quel Pokémon est de type Feu ?", "choix": ["Bulbizarre", "Salamèche", "Carapuce"], "bonne_reponse": "Salamèche"},
]
reponses_joueur = ["électrique", "2", "Eau"]

def corriger(q, reponse_joueur):
    reponse = reponse_joueur.strip()
    if reponse.isdigit() and 1 <= int(reponse) <= len(q["choix"]):     # réponse par numéro
        reponse = q["choix"][int(reponse) - 1]
    return reponse.lower() == q["bonne_reponse"].lower()

def jouer(banque, reponses):
    details = [corriger(q, r) for q, r in zip(banque, reponses)]
    return {"score": sum(details), "total": len(banque), "details": details}

bilan = jouer(banque, reponses_joueur)
bilan_json = json.dumps(bilan, ensure_ascii=False)
print(bilan_json)   # {"score": 2, "total": 3, "details": [true, true, false]}
# Pour aller plus loin : génère les quiz avec demander_json + valider_quiz au lieu de la banque écrite à la main.
```

</details>

## Bravo !

Tu sais maintenant construire des messages corrects, écrire une fiche de poste, garder (et tronquer) l'historique, et surtout obtenir du **JSON fiable** : extraire, valider, redemander. C'est exactement ce dont ton bot à thème a besoin pour compter des points ou gérer un inventaire.